In [1]:
# goal: detect family in photos

from fx import (
    load_image_file,  
    face_encodings,     # face-recognition/dlib image -> array 
    get_encodings,      # get list of face encodings 
    ensure_lst,
    # save_encodings
    np,pd
)

In [2]:
pics = [
    'labeled_pics/IMG_0210-2016-11-06 211542.JPG',  # should find 2 faces
    'labeled_pics/IMG_0025-2017-12-26 141137.JPG'   # should find 1 face
]
# for p in pics: show(p)

for p in pics:
    img = load_image_file(p)  # uses PIL
    encodings = face_encodings(img)
    print('faces found: ',len(encodings))


faces found:  0
faces found:  0


In [3]:
for p in pics:
    # get_encodings uses opencv instead of PIL
    # and reduces size for speed
    print('faces found: ',len(get_encodings(p)))

faces found:  2
faces found:  1


In [4]:
# curated some pics with just one face to use as labels
def label(pics:list):
    pics = ensure_lst(pics)
    return [
        get_encodings(i)
        [0] # get one & only fac in pic
        for i in pics
    ]

wife = label('labeled_pics/wife.JPG')
me = label(['labeled_pics/me.JPG'])
son = label(['labeled_pics/IMG_0025-2017-12-26 141137.JPG'])
daughter = label([
        'labeled_pics/daughter.JPG',  
        'labeled_pics/daughter1.JPG',
])

print(f'each face encodes to array of {len(wife[0])} length ')

# save_encodings("labeled_embeddings/daughter.npz",daughter)
# save_encodings("labeled_embeddings/me.npz",me)
# save_encodings('labeled_embeddings/son.npz',son)

each face encodes to array of 128 length 


In [5]:
# testing
def face_matches(known_faces,faces,tolerance=.6):
    known_faces=ensure_lst(known_faces)
    faces=ensure_lst(faces)    
    for f in faces:
        # print('face shape',f.shape)
        for dist in np.linalg.norm(known_faces - f, axis=1):
            # print('dist ',dist)
            if dist < tolerance: return 1
    return 0

print('does daughter at 18 months match 3 years? ',face_matches(daughter[0],daughter[1]),'\n')
print('daughter match wife? ',face_matches(daughter,wife),'\n')
print('me match wife? ',face_matches(me,wife))

does daughter at 18 months match 3 years?  1 

daughter match wife?  0 

me match wife?  0


In [6]:
# should match on me & daughter
f = 'labeled_pics/DSC_0010-2018-02-23 065122.JPG'
# show(f)
assert face_matches(daughter,get_encodings(f))
assert face_matches(me,get_encodings(f))
assert not face_matches(wife,get_encodings(f))
print('pass')

pass


In [7]:
# does this pic up son & daughter
f = 'labeled_pics/IMG_0210-2016-11-06 211542.JPG'

# show(f)
assert face_matches(daughter,get_encodings(f))
assert face_matches(son,get_encodings(f))
assert not face_matches(me,get_encodings(f))
assert not face_matches(wife,get_encodings(f))
assert face_matches(wife,get_encodings('labeled_pics/June 2008 035.jpg'))
print('pass')

pass


In [8]:
me_with_beard='labeled_pics/IMG_E1842-2021-09-14 025917.JPG'
assert face_matches(me,get_encodings(me_with_beard))
# assert not face_matches(wife,get_encodings(me_with_beard))  # fail :(
print('done')

done


In [15]:
def all_matches(f):
    encodings = get_encodings(f)
    return {
        'path': f,
        'wife': face_matches(wife,encodings),
        'daughter': face_matches(daughter,encodings),
        'son': face_matches(son,encodings),
        'me': face_matches(me,encodings),
    }    

# 14 second to do 60 images => 45 minutes for whole?

results = []
files = list(
    pd.read_parquet('features/files.parquet')
    # .sample(25)
    .path)
for f in files:
    results.append(all_matches(f))

results_file='features/faces.parquet'
(
    pd.DataFrame(results)
    .to_parquet(results_file,index=False)
)

del results
(
    pd.read_parquet(results_file)
    .print_shape()
    .sample(20)
)

(13652, 5)


,path,wife,daughter,son,me
43,/mnt/4C74F47B74F468DA/Pictures/IMG_1483-2021-0...,0,1,1,0
1455,/mnt/4C74F47B74F468DA/Pictures/IMG_2378-2018-0...,0,0,0,0
6076,/mnt/4C74F47B74F468DA/Pictures/IMG_7372-2021-1...,1,1,0,0
9623,/mnt/4C74F47B74F468DA/Pictures/DSC_0887-2016-0...,0,0,0,0
3409,/mnt/4C74F47B74F468DA/Pictures/IMG_1256-2018-0...,0,0,0,0
4668,/mnt/4C74F47B74F468DA/Pictures/IMG_3852-2015-0...,0,0,0,0
11169,/mnt/4C74F47B74F468DA/Pictures/DSC_0687-2013-1...,0,0,0,0
4078,/mnt/4C74F47B74F468DA/Pictures/IMG_21042013-06...,0,0,0,0
12049,/mnt/4C74F47B74F468DA/Pictures/disney/nathan/I...,1,1,0,0
1758,/mnt/4C74F47B74F468DA/Pictures/IMG_2741-2014-0...,0,0,0,0


In [ ]:

#7 seconds for 5 files


#should match wife & does
f = '/mnt/4C74F47B74F468DA/Pictures/disney/laurens/IMG_6160.JPG'

# wife = get_encodings('labeled_pics/wife.JPG',jitters=25)

# matches wife but shouldnt
f = "/mnt/4C74F47B74F468DA/Pictures/IMG_1900-2018-05-21 131315.JPG"
# show(f)
# enc=get_encodings(f,jitters=25)
# face_matches(me,get_encodings(f))
# face_matches(daughter,get_encodings(f))
# face_matches(wife,get_encodings(f))

# pd.read_parquet('features/unique.parquet').shape


# 14 / 60  # 14s to do 60 images
# print('hours = ',10109 / (60 *60)) 


